In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install(
    "ultralytics",
    "scikit-learn",
    "matplotlib",
    "seaborn",
    "opencv-python-headless",
    "filterpy",
    "lap",
    "sahi",
    "PyYAML",
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, cv2, shutil, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import defaultdict
from sklearn.model_selection import train_test_split
import yaml
import torch
from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment
from filterpy.kalman import KalmanFilter
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT_DIR        = '/content/drive/MyDrive/DriveIndia'
IMAGE_DIR       = f'{ROOT_DIR}/images'
LABEL_DIR       = f'{ROOT_DIR}/labels'
TRAIN_IMAGE_DIR = f'{IMAGE_DIR}/train'
VAL_IMAGE_DIR   = f'{IMAGE_DIR}/val'
TRAIN_LABEL_DIR = f'{LABEL_DIR}/train'
VAL_LABEL_DIR   = f'{LABEL_DIR}/val'
EXPERIMENT_DIR  = f'{ROOT_DIR}/driveindia_experiment'
TRACKING_DIR    = f'{ROOT_DIR}/tracking_results'
DATASET_YAML    = f'{ROOT_DIR}/dataset.yaml'
BEST_MODEL_PATH = f'{EXPERIMENT_DIR}/yolov8m_driveindia/weights/best.pt'

for d in [EXPERIMENT_DIR, TRACKING_DIR,
          TRAIN_IMAGE_DIR, VAL_IMAGE_DIR,
          TRAIN_LABEL_DIR, VAL_LABEL_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Root   : {ROOT_DIR}")
print(f"Images : {IMAGE_DIR}")
print(f"Labels : {LABEL_DIR}")


Root   : /content/drive/MyDrive/DriveIndia
Images : /content/drive/MyDrive/DriveIndia/images
Labels : /content/drive/MyDrive/DriveIndia/labels


In [ ]:
RAW_CLASS_MAPPING = {
    # ── Person ──────────────────────────────
    '0':  'Person',

    # ── Vehicles ────────────────────────────
    '1':  'Bicycle',
    '2':  'Car',
    '3':  'Motorcycle',
    '4':  'Bus',
    '5':  'Commercial vehicle',
    '6':  'Truck',
    '7':  'Autorickshaw',
    '8':  'Ambulance',
    '9':  'Police vehicle',
    '10': 'Tractor',
    '11': 'Pushcart',
    '12': 'Construction vehicle',

    # ── Road Infrastructure ──────────────────
    '13': 'Route board',
    '14': 'Traffic sign',
    '15': 'Traffic light',
    '16': 'Temporary traffic barrier',
    '17': 'Traffic cone',
    '18': 'Rumblestrips',
    '19': 'Unmarked speed bump',
    '20': 'Marked speed bump',
    '21': 'Zebra crossing',

    # ── Environmental & Surface Anomalies ────
    '22': 'Animal',
    '23': 'Pothole',
}

sorted_orig_ids  = sorted(RAW_CLASS_MAPPING.keys(), key=lambda x: int(x))
id_mapping       = {orig: str(new) for new, orig in enumerate(sorted_orig_ids)}
CLASS_NAMES      = [RAW_CLASS_MAPPING[oid] for oid in sorted_orig_ids]
NUM_CLASSES      = len(CLASS_NAMES)

# ── Small / fast-moving classes that benefit most from SAHI ───────────────────
SMALL_CLASSES = {
    'Bicycle', 'Motorcycle', 'Autorickshaw', 'Pushcart',
    'Traffic cone', 'Pothole', 'Person',
}
SMALL_CLASS_IDS = {i for i, n in enumerate(CLASS_NAMES) if n in SMALL_CLASSES}

print(f"\n✓ {NUM_CLASSES} classes configured")
for new_id, name in enumerate(CLASS_NAMES):
    tag = '⬡ small' if new_id in SMALL_CLASS_IDS else ''
    print(f"  {new_id:2d}: {name} {tag}")




✓ 24 classes configured
   0: Person ⬡ small
   1: Bicycle ⬡ small
   2: Car 
   3: Motorcycle ⬡ small
   4: Bus 
   5: Commercial vehicle 
   6: Truck 
   7: Autorickshaw ⬡ small
   8: Ambulance 
   9: Police vehicle 
  10: Tractor 
  11: Pushcart ⬡ small
  12: Construction vehicle 
  13: Route board 
  14: Traffic sign 
  15: Traffic light 
  16: Temporary traffic barrier 
  17: Traffic cone ⬡ small
  18: Rumblestrips 
  19: Unmarked speed bump 
  20: Marked speed bump 
  21: Zebra crossing 
  22: Animal 
  23: Pothole ⬡ small


In [ ]:
valid_orig_ids  = set(RAW_CLASS_MAPPING.keys())
files_kept      = 0
files_removed   = 0
annotation_counts = defaultdict(int)

for label_file in os.listdir(LABEL_DIR):
    if not label_file.endswith('.txt'):
        continue

    file_path = os.path.join(LABEL_DIR, label_file)
    with open(file_path, 'r') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
        old_id = parts[0]
        if old_id in valid_orig_ids:
            parts[0] = id_mapping[old_id]
            new_lines.append(' '.join(parts))
            annotation_counts[int(parts[0])] += 1

    if new_lines:
        with open(file_path, 'w') as f:
            f.write('\n'.join(new_lines))
        files_kept += 1
    else:
        os.remove(file_path)
        files_removed += 1

print(f"\nLabel filtering complete:")
print(f"  Kept   : {files_kept} files")
print(f"  Removed: {files_removed} files (no matching annotations)")
print("\nAnnotation counts per class:")
for new_id, name in enumerate(CLASS_NAMES):
    cnt = annotation_counts.get(new_id, 0)
    print(f"  {new_id:2d} {name:<28}: {cnt:>6}")




Label filtering complete:
  Kept   : 0 files
  Removed: 0 files (no matching annotations)

Annotation counts per class:
   0 Person                      :      0
   1 Bicycle                     :      0
   2 Car                         :      0
   3 Motorcycle                  :      0
   4 Bus                         :      0
   5 Commercial vehicle          :      0
   6 Truck                       :      0
   7 Autorickshaw                :      0
   8 Ambulance                   :      0
   9 Police vehicle              :      0
  10 Tractor                     :      0
  11 Pushcart                    :      0
  12 Construction vehicle        :      0
  13 Route board                 :      0
  14 Traffic sign                :      0
  15 Traffic light               :      0
  16 Temporary traffic barrier   :      0
  17 Traffic cone                :      0
  18 Rumblestrips                :      0
  19 Unmarked speed bump         :      0
  20 Marked speed bump           :     

In [ ]:
valid_label_names = set(os.listdir(LABEL_DIR))

image_files = [
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    and f.rsplit('.', 1)[0] + '.txt' in valid_label_names
]

print(f"Valid image-label pairs: {len(image_files)}")

train_files, val_files = train_test_split(
    image_files, test_size=0.2, random_state=42
)
print(f"Train: {len(train_files)}  |  Val: {len(val_files)}")


def _move_pair(img_file, src_img_dir, dst_img_dir, src_lbl_dir, dst_lbl_dir):
    """Move image + corresponding label file to destination directories."""
    stem = img_file.rsplit('.', 1)[0]

    src_img = os.path.join(src_img_dir, img_file)
    dst_img = os.path.join(dst_img_dir, img_file)
    if os.path.exists(src_img) and not os.path.exists(dst_img):
        shutil.move(src_img, dst_img)

    lbl_file = stem + '.txt'
    src_lbl = os.path.join(src_lbl_dir, lbl_file)
    dst_lbl = os.path.join(dst_lbl_dir, lbl_file)
    if os.path.exists(src_lbl) and not os.path.exists(dst_lbl):
        shutil.move(src_lbl, dst_lbl)


print("Moving training files …")
for f in train_files:
    _move_pair(f, IMAGE_DIR, TRAIN_IMAGE_DIR, LABEL_DIR, TRAIN_LABEL_DIR)

print("Moving validation files …")
for f in val_files:
    _move_pair(f, IMAGE_DIR, VAL_IMAGE_DIR, LABEL_DIR, VAL_LABEL_DIR)

print("✓ Split complete")



Valid image-label pairs: 0


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [ ]:
dataset_config = {
    'path' : ROOT_DIR,
    'train': 'images/train',
    'val'  : 'images/val',
    'nc'   : NUM_CLASSES,
    'names': CLASS_NAMES,
}

with open(DATASET_YAML, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False, allow_unicode=True)

print("dataset.yaml written:")
print(yaml.dump(dataset_config, default_flow_style=False))




In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU   : {torch.cuda.get_device_name(0)}")

# Using the ultralytics Python API instead of shell command —
# avoids quoting / path issues inside Colab.
model = YOLO('yolov8m.pt')

model.train(
    data           = DATASET_YAML,
    epochs         = 60,
    imgsz          = 640,
    batch          = 16,
    device         = 0 if device == 'cuda' else 'cpu',
    workers        = 8,
    patience       = 30,
    cos_lr         = True,
    lr0            = 0.01,
    lrf            = 0.01,
    momentum       = 0.937,
    weight_decay   = 0.0005,
    warmup_epochs  = 3,
    warmup_momentum= 0.8,
    warmup_bias_lr = 0.1,
    box            = 7.5,
    cls            = 0.5,
    dfl            = 1.5,
    hsv_h          = 0.015,
    hsv_s          = 0.7,
    hsv_v          = 0.4,
    degrees        = 10.0,
    translate      = 0.2,
    scale          = 0.9,
    flipud         = 0.0,
    fliplr         = 0.5,
    mosaic         = 1.0,
    mixup          = 0.15,
    copy_paste     = 0.3,
    project        = EXPERIMENT_DIR,
    name           = 'yolov8m_driveindia',
    exist_ok       = True,
    verbose        = True,
)

print(f"\nTraining complete — best model: {BEST_MODEL_PATH}")




In [ ]:
val_model = YOLO(BEST_MODEL_PATH)
results   = val_model.val(data=DATASET_YAML, conf=0.25, iou=0.5, max_det=300)

print(f"\nOverall Results:")
print(f"  mAP50   : {results.box.map50:.3f}")
print(f"  mAP50-95: {results.box.map:.3f}")
print(f"  Precision: {results.box.mp:.3f}")
print(f"  Recall   : {results.box.mr:.3f}")

print("\n" + "─" * 68)
print(f"{'Class':<28} {'mAP50':>8} {'mAP50-95':>9} {'Prec':>8} {'Recall':>8}")
print("─" * 68)

ap_class_list = results.box.ap_class_index.tolist()
for i, name in enumerate(CLASS_NAMES):
    if i in ap_class_list:
        idx  = ap_class_list.index(i)
        ap50 = results.box.ap50[idx]
        ap   = results.box.ap[idx]
        p    = results.box.p[idx]
        r    = results.box.r[idx]
        print(f"{name:<28} {ap50:8.3f} {ap:9.3f} {p:8.3f} {r:8.3f}")

print("─" * 68)



In [ ]:
class KalmanBoxTracker:
    """Single-object Kalman filter tracker using [cx, cy, w, h, vx, vy, vs] state."""

    count = 0  # class-level counter — reset externally when needed

    def __init__(self, bbox):
        """bbox: [x1, y1, x2, y2]"""
        KalmanBoxTracker.count += 1
        self.track_id           = KalmanBoxTracker.count
        self.hits               = 1
        self.hit_streak         = 1
        self.time_since_update  = 0
        self.age                = 0
        self.history            = []

        self.kf = KalmanFilter(dim_x=7, dim_z=4)

        # State-transition
        self.kf.F = np.array([
            [1,0,0,0,1,0,0],
            [0,1,0,0,0,1,0],
            [0,0,1,0,0,0,1],
            [0,0,0,1,0,0,0],
            [0,0,0,0,1,0,0],
            [0,0,0,0,0,1,0],
            [0,0,0,0,0,0,1],
        ], dtype=np.float32)

        # Measurement matrix
        self.kf.H = np.eye(4, 7, dtype=np.float32)

        self.kf.R[2:, 2:]  *= 10.0   # measurement noise
        self.kf.P[4:, 4:]  *= 1000.0  # high uncertainty on velocities
        self.kf.P          *= 10.0
        self.kf.Q[-1, -1]  *= 0.01
        self.kf.Q[4:, 4:]  *= 0.01

        x, y, x2, y2 = bbox
        cx = (x + x2) / 2
        cy = (y + y2) / 2
        w  = x2 - x
        h  = y2 - y
        s  = w * h          # scale (area)
        r  = w / float(h) if h > 0 else 1.0

        self.kf.x[:4] = [[cx], [cy], [s], [r]]
        self.last_obs  = np.array([cx, cy, w, h])
        self.velocity  = np.zeros(2)

    # ── helpers ──────────────────────────────────────────────────────────────
    @staticmethod
    def _xyxy_to_ccwh(bbox):
        x1, y1, x2, y2 = bbox
        cx = (x1 + x2) / 2; cy = (y1 + y2) / 2
        w  = x2 - x1;        h  = y2 - y1
        return cx, cy, w, h

    @staticmethod
    def _state_to_xyxy(x):
        cx, cy, s, r = float(x[0]), float(x[1]), float(x[2]), float(x[3])
        if s < 0: s = 0
        w = np.sqrt(s * abs(r)) if r > 0 else 0
        h = s / w if w > 0 else 0
        x1 = cx - w / 2; y1 = cy - h / 2
        return np.array([x1, y1, x1 + w, y1 + h])

    # ── public API ────────────────────────────────────────────────────────────
    def predict(self):
        if (self.kf.x[6] + self.kf.x[2]) <= 0:
            self.kf.x[6] = 0.0
        self.kf.predict()
        self.age += 1
        if self.time_since_update > 0:
            self.hit_streak = 0
        self.time_since_update += 1
        pred = self._state_to_xyxy(self.kf.x)
        self.history.append(pred)
        return pred

    def update(self, bbox):
        """bbox: [x1, y1, x2, y2]"""
        self.time_since_update = 0
        self.hits             += 1
        self.hit_streak       += 1
        self.history           = []

        cx, cy, w, h = self._xyxy_to_ccwh(bbox)
        s = w * h
        r = w / float(h) if h > 0 else 1.0

        # OC-SORT: velocity update
        new_obs = np.array([cx, cy])
        self.velocity  = new_obs - self.last_obs[:2]
        self.last_obs  = np.array([cx, cy, w, h])

        self.kf.update([[cx], [cy], [s], [r]])

    def get_state(self):
        return self._state_to_xyxy(self.kf.x)


def _iou_batch(bb_test: np.ndarray, bb_gt: np.ndarray) -> np.ndarray:
    """Vectorised IoU between two sets of boxes [N,4] vs [M,4]."""
    bb_gt = np.expand_dims(bb_gt, 0)   # [1,M,4]
    bb_test = np.expand_dims(bb_test, 1) # [N,1,4]

    xx1 = np.maximum(bb_test[...,0], bb_gt[...,0])
    yy1 = np.maximum(bb_test[...,1], bb_gt[...,1])
    xx2 = np.minimum(bb_test[...,2], bb_gt[...,2])
    yy2 = np.minimum(bb_test[...,3], bb_gt[...,3])

    inter = np.maximum(0, xx2 - xx1) * np.maximum(0, yy2 - yy1)
    a1    = (bb_test[...,2]-bb_test[...,0]) * (bb_test[...,3]-bb_test[...,1])
    a2    = (bb_gt[...,2]  -bb_gt[...,0])   * (bb_gt[...,3]  -bb_gt[...,1])
    union = a1 + a2 - inter
    return inter / np.where(union == 0, 1e-9, union)


class OCSort:
    """
    OC-SORT: Observation-Centric SORT tracker.
    Accepts detections as [x1, y1, x2, y2, score] arrays.
    Returns tracked objects as [x1, y1, x2, y2, track_id] arrays.
    """

    def __init__(self,
                 det_thresh    = 0.30,
                 max_age       = 30,
                 min_hits      = 3,
                 iou_threshold = 0.30):
        self.det_thresh    = det_thresh
        self.max_age       = max_age
        self.min_hits      = min_hits
        self.iou_threshold = iou_threshold
        self.trackers      = []
        self.frame_count   = 0
        KalmanBoxTracker.count = 0  # reset global counter

    # ── internal helpers ──────────────────────────────────────────────────────
    def _associate(self, detections: np.ndarray, trackers: np.ndarray):
        """
        detections : (N,5)  [x1,y1,x2,y2,score]
        trackers   : (M,5)  [x1,y1,x2,y2,track_id]
        Returns matched, unmatched_dets, unmatched_trks  (all index lists)
        """
        if trackers.shape[0] == 0:
            return [], list(range(len(detections))), []
        if len(detections) == 0:
            return [], [], list(range(len(trackers)))

        iou_matrix = _iou_batch(detections[:, :4], trackers[:, :4])  # [N,M]
        row_idx, col_idx = linear_sum_assignment(-iou_matrix)

        unmatched_dets = [d for d in range(len(detections)) if d not in row_idx]
        unmatched_trks = [t for t in range(len(trackers))   if t not in col_idx]

        matched = []
        for r, c in zip(row_idx, col_idx):
            if iou_matrix[r, c] < self.iou_threshold:
                unmatched_dets.append(r)
                unmatched_trks.append(c)
            else:
                matched.append((r, c))   # (det_idx, trk_idx)

        return matched, unmatched_dets, unmatched_trks

    # ── public API ────────────────────────────────────────────────────────────
    def update(self, dets: np.ndarray) -> np.ndarray:
        """
        dets: np.ndarray shape (N, 5) → [x1, y1, x2, y2, score]
              or (0, 5) / empty array for frames with no detections.

        Returns: np.ndarray shape (M, 5) → [x1, y1, x2, y2, track_id]
        """
        self.frame_count += 1

        # ── 1. Predict from existing trackers ─────────────────────────────────
        trk_preds = np.zeros((len(self.trackers), 5))
        to_del    = []
        for t, trk in enumerate(self.trackers):
            pred = trk.predict()
            trk_preds[t, :4] = pred
            trk_preds[t,  4] = trk.track_id
            if np.any(np.isnan(pred)):
                to_del.append(t)
        for t in reversed(to_del):
            self.trackers.pop(t)
            trk_preds = np.delete(trk_preds, t, axis=0)

        # ── 2. Filter low-confidence detections ──────────────────────────────
        if dets is None or len(dets) == 0:
            dets = np.empty((0, 5))
        dets = np.array(dets)
        if dets.ndim == 1:
            dets = dets.reshape(1, -1)
        high_conf = dets[dets[:, 4] >= self.det_thresh] if len(dets) > 0 else dets

        # ── 3. Associate ──────────────────────────────────────────────────────
        matched, unmatched_dets, unmatched_trks = self._associate(
            high_conf, trk_preds
        )

        # ── 4. Update matched trackers ────────────────────────────────────────
        for det_idx, trk_idx in matched:
            self.trackers[trk_idx].update(high_conf[det_idx, :4])

        # ── 5. Spawn new trackers for unmatched high-conf dets ────────────────
        for det_idx in unmatched_dets:
            self.trackers.append(KalmanBoxTracker(high_conf[det_idx, :4]))

        # ── 6. Remove dead trackers ───────────────────────────────────────────
        self.trackers = [
            t for t in self.trackers if t.time_since_update <= self.max_age
        ]

        # ── 7. Build output ───────────────────────────────────────────────────
        output = []
        for trk in self.trackers:
            if (trk.time_since_update < 1 and
                    (trk.hit_streak >= self.min_hits or
                     self.frame_count <= self.min_hits)):
                state = trk.get_state()
                output.append([*state, trk.track_id])

        return np.array(output, dtype=np.float32) if output else np.empty((0, 5))


print("✓ OC-SORT tracker implemented")




In [ ]:
class DriveIndiaTracker:
    """
    High-level wrapper that couples YOLO detections (per class) with
    one OC-SORT tracker per class so identities are stable.
    """

    def __init__(self,
                 class_names   : list,
                 det_thresh    : float = 0.30,
                 max_age       : int   = 30,
                 min_hits      : int   = 3,
                 iou_threshold : float = 0.30):
        self.class_names = class_names
        self.trackers    = {
            i: OCSort(det_thresh=det_thresh,
                      max_age=max_age,
                      min_hits=min_hits,
                      iou_threshold=iou_threshold)
            for i in range(len(class_names))
        }
        self.track_trails = defaultdict(list)  # track_id → list of (cx,cy)
        self.frame_count  = 0

    def update(self, yolo_result) -> list:
        """
        yolo_result : single ultralytics Results object (one frame).
        Returns     : list of dicts with keys bbox, track_id, class_id, class_name
        """
        # Group raw detections by class
        class_dets = defaultdict(list)  # class_id → [[x1,y1,x2,y2,score], …]

        if yolo_result.boxes is not None and len(yolo_result.boxes):
            for box in yolo_result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                score           = float(box.conf[0])
                cls             = int(box.cls[0])
                class_dets[cls].append([x1, y1, x2, y2, score])

        # Update each class tracker
        tracked_objects = []
        for cls_id, dets in class_dets.items():
            dets_arr   = np.array(dets, dtype=np.float32)
            tracked    = self.trackers[cls_id].update(dets_arr)

            for row in tracked:
                x1, y1, x2, y2, tid = row
                tid = int(tid)
                cx  = (x1 + x2) / 2
                cy  = (y1 + y2) / 2
                self.track_trails[tid].append((cx, cy))
                if len(self.track_trails[tid]) > 50:
                    self.track_trails[tid] = self.track_trails[tid][-50:]

                tracked_objects.append({
                    'track_id'  : tid,
                    'bbox'      : [int(x1), int(y1), int(x2), int(y2)],
                    'class_id'  : cls_id,
                    'class_name': self.class_names[cls_id]
                        if cls_id < len(self.class_names) else 'Unknown',
                    'frame'     : self.frame_count,
                })

        self.frame_count += 1
        return tracked_objects

    def reset(self):
        for trk in self.trackers.values():
            trk.__init__(trk.det_thresh, trk.max_age,
                         trk.min_hits, trk.iou_threshold)
        self.track_trails.clear()
        self.frame_count = 0


print("✓ DriveIndiaTracker ready")



In [ ]:
def build_sahi_model(model_path: str, conf: float = 0.25) -> AutoDetectionModel:
    """Load the trained YOLO model into SAHI."""
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    return AutoDetectionModel.from_pretrained(
        model_type            = 'yolov8',
        model_path            = model_path,
        confidence_threshold  = conf,
        device                = device,
    )


def compute_iou(box1, box2) -> float:
    x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
    inter  = max(0, x2-x1) * max(0, y2-y1)
    area1  = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2  = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union  = area1 + area2 - inter
    return inter / union if union > 0 else 0.0


def run_vanilla_yolo(yolo_model, image_path: str,
                     conf: float = 0.25, iou: float = 0.50) -> list:
    """Run standard YOLO inference. Returns list of [x1,y1,x2,y2,score,cls]."""
    results = yolo_model(image_path, conf=conf, iou=iou)
    dets    = []
    if results[0].boxes is not None:
        for box in results[0].boxes:
            x1,y1,x2,y2 = box.xyxy[0].tolist()
            dets.append([x1, y1, x2, y2, float(box.conf[0]), int(box.cls[0])])
    return dets


def run_sahi_yolo(sahi_model, image_path: str,
                  slice_h: int = 640, slice_w: int = 640,
                  overlap: float = 0.2) -> list:
    """Run SAHI sliced inference. Returns list of [x1,y1,x2,y2,score,cls]."""
    result = get_sliced_prediction(
        image_path,
        sahi_model,
        slice_height              = slice_h,
        slice_width               = slice_w,
        overlap_height_ratio      = overlap,
        overlap_width_ratio       = overlap,
        postprocess_match_metric  = "IOU",
        postprocess_match_threshold = 0.5,
        verbose                   = 0,
    )
    dets = []
    for pred in result.object_prediction_list:
        bbox = pred.bbox.to_xyxy()
        dets.append([bbox[0], bbox[1], bbox[2], bbox[3],
                     pred.score.value, pred.category.id])
    return dets


print("✓ SAHI utilities ready")



In [ ]:
PALETTE = [
    (255, 56,  56 ), (255, 157, 151), (255, 112, 31 ), (255, 178, 29 ),
    (207, 210, 49 ), (72,  249, 10 ), (146, 204, 23 ), (61,  219, 134),
    (26,  147, 52 ), (0,   212, 187), (44,  153, 168), (0,   194, 255),
    (52,  69,  147), (100, 115, 255), (0,   24,  236), (132, 56,  255),
    (82,  0,   133), (203, 56,  255), (255, 149, 200), (255, 55,  199),
    (50,  50,  50 ), (150, 150, 150), (200, 100, 0  ), (0,   100, 200),
]

def class_color(cls_id: int):
    return PALETTE[cls_id % len(PALETTE)]


def draw_detections(image_rgb: np.ndarray, detections: list,
                    class_names: list, small_ids: set) -> np.ndarray:
    """Draw bounding boxes + labels on an RGB image copy."""
    img = image_rgb.copy()
    for det in detections:
        x1, y1, x2, y2 = map(int, det[:4])
        score, cls      = det[4], int(det[5])
        color           = class_color(cls)
        thickness       = 3 if cls in small_ids else 2

        cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)

        name  = class_names[cls] if cls < len(class_names) else str(cls)
        label = f"{name} {score:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(img, (x1, y1 - th - 6), (x1 + tw + 2, y1), color, -1)
        cv2.putText(img, label, (x1 + 1, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1,
                    cv2.LINE_AA)
    return img


def draw_tracked(image_rgb: np.ndarray, tracked: list,
                 class_names: list, trails: dict) -> np.ndarray:
    """Draw OC-SORT tracked boxes + motion trails."""
    img = image_rgb.copy()
    for obj in tracked:
        x1, y1, x2, y2 = obj['bbox']
        tid    = obj['track_id']
        cls    = obj['class_id']
        color  = class_color(cls)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        label = f"#{tid} {obj['class_name']}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(img, (x1, y1 - th - 6), (x1 + tw + 2, y1), color, -1)
        cv2.putText(img, label, (x1 + 1, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1,
                    cv2.LINE_AA)

        # Draw trail
        trail = trails.get(tid, [])
        for i in range(1, len(trail)):
            pt1 = tuple(map(int, trail[i-1]))
            pt2 = tuple(map(int, trail[i]))
            alpha = i / len(trail)
            t_color = tuple(int(c * alpha) for c in color)
            cv2.line(img, pt1, pt2, t_color, 2)

    return img


print("✓ Visualisation utilities ready")



In [ ]:
def compare_single_image(image_path    : str,
                         yolo_model,
                         sahi_model,
                         class_names   : list,
                         small_ids     : set,
                         save_path     : str = None) -> tuple:
    """
    Show a side-by-side comparison: Vanilla YOLOv8 vs SAHI + YOLOv8.
    Highlights new small-vehicle detections found only by SAHI.
    Returns (vanilla_dets, sahi_dets).
    """
    image = cv2.imread(image_path)
    if image is None:
        print(f"[ERROR] Cannot read: {image_path}")
        return [], []
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w      = image.shape[:2]

    print(f"\n{'='*64}")
    print(f"Image : {os.path.basename(image_path)}  ({w}×{h})")
    print(f"{'='*64}")

    # ── Vanilla YOLO ──────────────────────────────────────────────────────────
    print("[1/2] Vanilla YOLOv8m …")
    vanilla_dets = run_vanilla_yolo(yolo_model, image_path)
    vanilla_small = [d for d in vanilla_dets if int(d[5]) in small_ids]
    print(f"      {len(vanilla_dets)} total  |  {len(vanilla_small)} small-class")

    # ── SAHI + YOLO ──────────────────────────────────────────────────────────
    print("[2/2] SAHI + YOLOv8m (sliced) …")
    sahi_dets  = run_sahi_yolo(sahi_model, image_path)
    sahi_small = [d for d in sahi_dets if int(d[5]) in small_ids]
    print(f"      {len(sahi_dets)} total  |  {len(sahi_small)} small-class")

    # ── Delta analysis ────────────────────────────────────────────────────────
    new_dets = []
    for sd in sahi_small:
        if not any(compute_iou(sd[:4], vd[:4]) > 0.5 for vd in vanilla_small):
            new_dets.append(sd)

    delta = len(new_dets)
    if len(vanilla_small) > 0:
        pct = delta / len(vanilla_small) * 100
        print(f"\n  → SAHI found {delta} extra small-class detections (+{pct:.1f}%)")
    else:
        print(f"\n  → SAHI found {delta} extra small-class detections (vanilla found 0)")

    # ── Visualise ─────────────────────────────────────────────────────────────
    vanilla_viz = draw_detections(image_rgb, vanilla_dets, class_names, small_ids)
    sahi_viz    = draw_detections(image_rgb, sahi_dets,    class_names, small_ids)

    # Circle new detections on SAHI panel
    for det in new_dets:
        x1, y1, x2, y2 = map(int, det[:4])
        cx = (x1 + x2) // 2; cy = (y1 + y2) // 2
        r  = max((x2-x1)//2, (y2-y1)//2, 15)
        cv2.circle(sahi_viz, (cx, cy), r, (255, 255, 0), 3)

    fig, axes = plt.subplots(1, 2, figsize=(22, 10))
    axes[0].imshow(vanilla_viz)
    axes[0].set_title(
        f"Vanilla YOLOv8m\n{len(vanilla_dets)} dets  |  {len(vanilla_small)} small",
        fontsize=13, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(sahi_viz)
    axes[1].set_title(
        f"SAHI + YOLOv8m\n{len(sahi_dets)} dets  |  {len(sahi_small)} small"
        + (f"  (+{delta} new ⭕)" if delta else ""),
        fontsize=13, fontweight='bold')
    axes[1].axis('off')

    if delta:
        axes[1].text(8, 28, f"⭕ {delta} new small detections",
                     color='yellow', fontsize=11, fontweight='bold',
                     bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  Saved → {save_path}")
    plt.show()

    return vanilla_dets, sahi_dets


print("✓ compare_single_image() ready")

